# Notebook 02: Persiapan Data, Normalisasi Slang & Pelabelan ABSA (Bab 3 Laporan AM552)
**Mata Kuliah**: AM552 - Data Science of IS (Program Studi S1 Sistem Informasi, FTI UKSW)  
**Kode Kelompok**: `682024046_KEL_DS`

---
## 3.1 Langkah Persiapan Teks Bahasa Indonesia & Multi-Label ABSA
1. **Pembersihan Teks & Normalisasi Karakter Berulang**: Menghapus URL, karakter non-alfanumerik, dan mereduksi huruf berulang (`"bagusssss"` -> `"bagus"`).
2. **Normalisasi Slang Bahasa Indonesia dengan Preservasi Negasi**: Mengubah kata tidak baku (`"gk"`, `"gak"`, `"tdk"` -> `"tidak"`, `"bgt"` -> `"banget"`, `"lemot"` -> `"lambat"`) menggunakan `slang_dictionary_id.json` tanpa menghapus kata negasi (`tidak`, `kurang`, `jangan`, `bukan`).
3. **Pelabelan 4 Dimensi Aspek Layanan (ABSA)**: Memetakan setiap ulasan ke aspek **A1 (Akurasi & Kecerdasan)**, **A2 (Harga & Limit Kuota)**, **A3 (Performa, UI & Server)**, dan **A4 (Bahasa Indonesia)**.

In [1]:
import json, re
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('../dataset/RAW_DATASET_GENAI_INDONESIA_AM522B.csv', low_memory=False)
with open('../03_data_preparation/slang_dictionary_id.json', 'r', encoding='utf-8') as f:
    slang_map = json.load(f)

def preprocess_text(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'http\S+|www\.\S+', ' ', text.lower())
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    tokens = re.findall(r'[a-z0-9]+', text)
    return ' '.join(slang_map.get(t, t) for t in tokens)

df['clean_content'] = df['content'].apply(preprocess_text)
df['sentiment_3class'] = df['score'].apply(lambda s: '0_Negative' if s<=2 else ('1_Neutral' if s==3 else '2_Positive'))
df['strat_key'] = df['app_name'] + '_' + df['sentiment_3class']

# 3.2 PEMBAGIAN DATA 80:20 STRATIFIED SPLIT SEBELUM TF-IDF FIT (ZERO DATA LEAKAGE)
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_content'], df['sentiment_3class'],
    test_size=0.20, stratify=df['strat_key'], random_state=42
)
print(f'Total Data Awal : {len(df):,} baris (100%)')
print(f'Data Latih (80%): {len(X_train):,} baris')
print(f'Data Uji (20%)  : {len(X_test):,} baris')

Total Data Awal : 71,221 baris (100%)
Data Latih (80%): 56,976 baris
Data Uji (20%)  : 14,245 baris